In [2]:
// ==== Setup ====

import java.util.*;

// wall-clock time of r, in milliseconds
long millis(Runnable r) {
    System.gc();            // start from a clean heap: less noise
    long start = System.nanoTime();
    r.run();
    return (System.nanoTime() - start) / 1_000_000;
}

// test helpers
void check(boolean ok, String what) {
    System.out.println((ok ? "ok    " : "FAIL  ") + what);
}

In [ ]:
// ==== From N16 ====

class BTNode<E> {
    E data;
    BTNode<E> left, right;

    BTNode(E data) { this(data, null, null); }

    BTNode(E data, BTNode<E> left, BTNode<E> right) {
        this.data = data;
        this.left = left;
        this.right = right;
    }
}

<E> void inorder(BTNode<E> node, List<E> out) {
    if (node == null) return;
    inorder(node.left, out);
    out.add(node.data);
    inorder(node.right, out);
}

// the tree as a string, written node(left, right)
<E> String shape(BTNode<E> node) {
    if (node == null) return "";
    if (node.left == null && node.right == null) return "" + node.data;
    return node.data + "(" + shape(node.left) + ", " + shape(node.right) + ")";
}

# Binary search trees

## Ordering property

A binary tree in which, for **every** node:
- all values in its **left** subtree are **smaller** than the node's value;
- all values in its **right** subtree are **larger**.

- Values must be comparable: numbers, strings, any `Comparable<T>`.
- Duplicates: not allowed here. Allowing them needs a fixed rule for where equal values go.

## Is it a BST?

Written as `node(left, right)`:

- `50(30(20, 40), 70(60, 80))`
- `50(30(20, 60), 70)`
- `50(, 70(, 80(, 90)))`
- `cat(ox, mouse)`: in alphabetical order? By length, ties broken alphabetically?

Inorder traversal of a BST: what comes out? Why?

# Searching

## contains

Binary search, on a tree: compare with the node, then go left or right. Never come back.

In [ ]:
//            55
//         /      \
//       30        78
//      /  \      /   \
//    20    42   63    97
//         /    /  \
//       35    58   70
//                    \
//                     72

BTNode<Integer> root = new BTNode<>(55,
    new BTNode<>(30, new BTNode<>(20), new BTNode<>(42, new BTNode<>(35), null)),
    new BTNode<>(78,
        new BTNode<>(63, new BTNode<>(58), new BTNode<>(70, null, new BTNode<>(72))),
        new BTNode<>(97)));

In [ ]:
boolean contains(BTNode<Integer> node, int value) {
    if (node == null) return false;
    if (value < node.data) return contains(node.left, value);
    if (value > node.data) return contains(node.right, value);
    return true;
}

check(contains(root, 58), "58");
check(!contains(root, 40), "40");

- Values examined when searching for `58`? For `40`?
- Cost: `O(H)`, where `H` is the height of the tree. Best and worst `H` for `N` nodes?
- An iterative `contains`: does it need a stack?

# Adding

## add

- Search for the value. Where the search falls off the tree, attach a new leaf.
- Where do `13`, `65`, `75` go?
- Is there ever more than one possible place for a new value?

In [ ]:
// returns the root of the subtree, after adding value to it
BTNode<Integer> add(BTNode<Integer> node, int value) {
    if (node == null) return new BTNode<>(value);
    if (value < node.data) node.left = add(node.left, value);
    else if (value > node.data) node.right = add(node.right, value);
    return node;                                    // equal: already in the tree
}

BTNode<Integer> build(List<Integer> values) {
    BTNode<Integer> tree = null;
    for (int v : values) tree = add(tree, v);
    return tree;
}

In [ ]:
List<Integer> running = List.of(55, 30, 78, 20, 42, 63, 97, 35, 58, 70, 72);
BTNode<Integer> t = build(running);
check(shape(t).equals(shape(root)), "same tree as the running example");

for (int v : List.of(13, 65, 75)) t = add(t, v);
System.out.println(shape(t));
check(shape(t).equals("55(30(20(13, ), 42(35, )), 78(63(58, 70(65, 72(, 75))), 97))"), "13, 65, 75");

List<Integer> sorted = new ArrayList<>();
inorder(t, sorted);
System.out.println(sorted);

- `add` returns a node, and the caller stores it. Why? What would go wrong with a `void add`?
- `running` is the level order of the tree. Would its preorder rebuild the same tree? Its inorder?

## Order of adding

Same values, different order: different shape, different height.

In [ ]:
// empty tree: -1, so that a leaf has height 0
<E> int height(BTNode<E> node) {
    if (node == null) return -1;
    return 1 + Math.max(height(node.left), height(node.right));
}

System.out.printf("%8s %8s %8s %8s%n", "N", "log2 N", "sorted", "shuffled");
Random rng = new Random(102);
for (int n = 500; n <= 4000; n *= 2) {
    List<Integer> values = new ArrayList<>();
    for (int i = 0; i < n; i++) values.add(i);
    int sortedHeight = height(build(values));
    Collections.shuffle(values, rng);
    int shuffledHeight = height(build(values));
    System.out.printf("%8d %8d %8d %8d%n", n, (int) (Math.log(n) / Math.log(2)), sortedHeight, shuffledHeight);
}

- Sorted input: height `N - 1`. Which sorting algorithm had the same problem, on the same input?
- Shuffled input: height close to a small multiple of `log₂ N`.
- `contains` and `add` cost `O(H)`: the shape decides between `O(log N)` and `O(N)`.

# Removing

## Three cases

1. **Leaf**: the parent's reference becomes `null`. E.g. `35`.
2. **One child**: the parent's reference skips to that child. E.g. `70`.
3. **Two children**: e.g. `55`. Two subtrees, one reference: neither of the above works.
   - Replace the value with its **inorder predecessor** (largest value in the left subtree) or its **inorder successor** (smallest value in the right subtree).
   - Then remove that value from the subtree it came from.

- Why does the predecessor keep the ordering property, in the place of `55`?
- How many children can the predecessor have?

In [ ]:
// largest value in the left subtree: one step left, then right as far as possible
int predecessor(BTNode<Integer> node) {
    BTNode<Integer> current = node.left;
    while (current.right != null) current = current.right;
    return current.data;
}

// returns the root of the subtree, after removing value from it
BTNode<Integer> remove(BTNode<Integer> node, int value) {
    if (node == null) return null;                  // not in the tree
    if (value < node.data) node.left = remove(node.left, value);
    else if (value > node.data) node.right = remove(node.right, value);
    else {
        if (node.left == null) return node.right;   // leaf, or right child only
        if (node.right == null) return node.left;   // left child only
        node.data = predecessor(node);
        node.left = remove(node.left, node.data);
    }
    return node;
}

In [ ]:
check(shape(remove(build(running), 35)).equals("55(30(20, 42), 78(63(58, 70(, 72)), 97))"), "leaf: 35");
check(shape(remove(build(running), 70)).equals("55(30(20, 42(35, )), 78(63(58, 72), 97))"), "one child: 70");
check(shape(remove(build(running), 55)).equals("42(30(20, 35), 78(63(58, 70(, 72)), 97))"), "two children: 55");
check(shape(remove(build(running), 40)).equals(shape(root)), "not in the tree: 40");

- Removing `55` with the successor instead: which tree comes out?
- Cost of `remove`?

# A generic BST class

- From `Integer` to any `T extends Comparable<T>`, as for `ComparableSorter`: `value < node.data` becomes `value.compareTo(node.data) < 0`.
- Public methods on the tree, private recursive ones on the nodes: the same split as `quickSort` and `quickSortRec`.
- `size` is a field, updated by `add` and `remove`.

In [ ]:
class BST<T extends Comparable<T>> {
    private class Node {
        T data;
        Node left, right;

        Node(T data) { this.data = data; }
    }

    private Node root;
    private int size;

    int size() { return size; }

    boolean contains(T value) { return contains(root, value); }

    private boolean contains(Node node, T value) {
        if (node == null) return false;
        int c = value.compareTo(node.data);
        if (c < 0) return contains(node.left, value);
        if (c > 0) return contains(node.right, value);
        return true;
    }

    void add(T value) { root = add(root, value); }

    private Node add(Node node, T value) {
        if (node == null) {
            size++;
            return new Node(value);
        }
        int c = value.compareTo(node.data);
        if (c < 0) node.left = add(node.left, value);
        else if (c > 0) node.right = add(node.right, value);
        return node;
    }

    void remove(T value) { root = remove(root, value); }

    private Node remove(Node node, T value) {
        if (node == null) return null;
        int c = value.compareTo(node.data);
        if (c < 0) node.left = remove(node.left, value);
        else if (c > 0) node.right = remove(node.right, value);
        else {
            if (node.left == null) { size--; return node.right; }
            if (node.right == null) { size--; return node.left; }
            node.data = predecessor(node);
            node.left = remove(node.left, node.data);   // size-- happens in this call
        }
        return node;
    }

    private T predecessor(Node node) {
        Node current = node.left;
        while (current.right != null) current = current.right;
        return current.data;
    }

    // values in increasing order: an inorder traversal
    List<T> toList() {
        List<T> out = new ArrayList<>();
        toList(root, out);
        return out;
    }

    private void toList(Node node, List<T> out) {
        if (node == null) return;
        toList(node.left, out);
        out.add(node.data);
        toList(node.right, out);
    }
}

# Sets and maps

## BST as a set

- `add`, `contains`, `remove`, `size`: the operations of `Set<E>`.
- No duplicates: adding a value already in the tree changes nothing.
- `toList`: the values in increasing order, an inorder traversal.

In [ ]:
String text = "the cat saw the dog and the dog saw the cat";

BST<String> words = new BST<>();
for (String w : text.split(" ")) words.add(w);

System.out.println(words.toList());
check(words.size() == 5, "5 distinct words");
check(words.contains("dog") && !words.contains("cow"), "contains");

- The same words in a `HashSet`: in which order do they come out?

## BST as a map

- Each node: a key **and** a value.
- Ordered by key only: `K extends Comparable<K>`; `V` can be any type.
- `get(k)`: as `contains`, returning the value of `k`.
- `put(k, v)`: as `add`; if `k` is found, replace its value.

In [ ]:
class BSTMap<K extends Comparable<K>, V> {
    private class Node {
        K key;
        V value;
        Node left, right;

        Node(K key, V value) {
            this.key = key;
            this.value = value;
        }
    }

    private Node root;
    private int size;

    int size() { return size; }

    // the value of key, or null if key is not in the map
    V get(K key) { return get(root, key); }

    private V get(Node node, K key) {
        if (node == null) return null;
        int c = key.compareTo(node.key);
        if (c < 0) return get(node.left, key);
        if (c > 0) return get(node.right, key);
        return node.value;
    }

    void put(K key, V value) { root = put(root, key, value); }

    private Node put(Node node, K key, V value) {
        if (node == null) {
            size++;
            return new Node(key, value);
        }
        int c = key.compareTo(node.key);
        if (c < 0) node.left = put(node.left, key, value);
        else if (c > 0) node.right = put(node.right, key, value);
        else node.value = value;                        // key already there: replace its value
        return node;
    }
}

In [ ]:
BSTMap<String, Integer> counts = new BSTMap<>();
for (String w : text.split(" ")) {
    Integer n = counts.get(w);
    counts.put(w, n == null ? 1 : n + 1);
}

check(counts.size() == 5, "5 keys");
check(counts.get("the") == 4 && counts.get("dog") == 2, "the: 4, dog: 2");
check(counts.get("cow") == null, "cow: not a key");

- `remove(k)`, with two children: what moves into the node, besides the key?
- `HashSet` is a `HashMap` with no values. A set from `BSTMap`?

## BST vs. hash table

| | hash table | BST |
|---|---|---|
| keys need | `equals`, `hashCode` | `compareTo`, or a `Comparator` |
| `get`, `put`, `remove` | `O(1)` average | `O(H)` |
| worst case | `O(N)`: all keys in one bucket | `O(N)`: `H = N - 1`, e.g. keys added in order |
| keys in order | sort them: `O(N log N)` | inorder: `O(N)` |
| smallest, largest key | `O(N)` | `O(H)` |
| largest key `≤ k` | `O(N)` | `O(H)` |
| memory per pair | a node: key, value, `next`; plus the array | a node: key, value, `left`, `right` |

- **Balanced** BST: `H` stays `O(log N)` after every `put` and `remove` (later in the course). Every operation in the BST column: `O(log N)` in the worst case.
- Largest key `≤ k` in `O(H)`: how?

## In `java.util`

| class | implements | |
|---|---|---|
| `TreeMap<K, V>` | `Map<K, V>` | a balanced BST |
| `TreeSet<E>` | `Set<E>` | a `TreeMap` with no values |

- Keys: `Comparable`, or a `Comparator` passed to the constructor.
- Iteration: keys in increasing order.
- Operations on the order: `firstKey`, `lastKey`, `floorKey`, `ceilingKey`, `headMap`, `tailMap`, ...

In [ ]:
Map<String, Integer> hash = new HashMap<>();
TreeMap<String, Integer> tree = new TreeMap<>();
for (String w : text.split(" ")) {
    hash.put(w, hash.getOrDefault(w, 0) + 1);
    tree.put(w, tree.getOrDefault(w, 0) + 1);
}

System.out.println("HashMap: " + hash);
System.out.println("TreeMap: " + tree);
check(hash.equals(tree), "same pairs");

System.out.println(tree.firstKey() + ", " + tree.lastKey());
System.out.println(tree.floorKey("cow") + ", " + tree.ceilingKey("cow"));
System.out.println(tree.headMap("dog"));        // keys < "dog"

## Equal keys

- A `TreeSet` never calls `equals`: two keys are the same key when `compareTo`, or the `Comparator`, returns `0`.
- What does this print? And with a `HashSet`?

In [ ]:
TreeSet<String> cities = new TreeSet<>(String.CASE_INSENSITIVE_ORDER);
cities.add("Rome");
cities.add("rome");
cities.add("ROME");
System.out.println(cities + ", size " + cities.size());

## Choosing

Hash table or BST?

- Count the words of a book, then print them in alphabetical order.
- Check whether a NetID is registered.
- Find the first train leaving after 14:35.
- Keys of a class with `equals` and `hashCode`, but no natural order.

## Exercises

### 1. Build and search
Insert, into an empty BST: `42, 21, 64, 10, 32, 54, 75, 28, 37, 50, 59, 26`.

- Draw the tree.
- Trace the search for `37` and for `52`: which values are examined?
- Smallest and largest values?
- Fifth smallest value?

`42(21(10, 32(28(26, ), 37)), 64(54(50, 59), 75))`

- `37`: `42, 21, 32, 37`, found. `52`: `42, 64, 54, 50`, then `50` has no right child: not found.
- Smallest `10` (leftmost node), largest `75` (rightmost node).
- In increasing order (inorder): `10, 21, 26, 28, 32, 37, 42, 50, 54, 59, 64, 75`. Fifth: `32`.

In [ ]:
List<Integer> ex1 = List.of(42, 21, 64, 10, 32, 54, 75, 28, 37, 50, 59, 26);
check(shape(build(ex1)).equals("42(21(10, 32(28(26, ), 37)), 64(54(50, 59), 75))"), "shape");

### 2. Insertion order and tree shape
Insert each sequence into a separate, empty BST.

- A: `40, 20, 60, 10, 30, 50, 70`
- B: `10, 20, 30, 40, 50, 60, 70`
- C: `40, 10, 60, 20, 30, 50, 70`

For each tree: draw it, find its height, count the values examined when searching for `10` and for `70`. Which order gives the most efficient searches? Why?

| | tree | height | `10` | `70` |
|---|---|---|---|---|
| A | `40(20(10, 30), 60(50, 70))` | 2 | 3 | 3 |
| B | `10(, 20(, 30(, 40(, 50(, 60(, 70))))))` | 6 | 1 | 7 |
| C | `40(10(, 20(, 30)), 60(50, 70))` | 3 | 2 | 3 |

A: height `2 = ⌊log₂ 7⌋`, the smallest possible for 7 nodes. Every search examines at most 3 values.

In [ ]:
check(height(build(List.of(40, 20, 60, 10, 30, 50, 70))) == 2, "A");
check(height(build(List.of(10, 20, 30, 40, 50, 60, 70))) == 6, "B");
check(height(build(List.of(40, 10, 60, 20, 30, 50, 70))) == 3, "C");

### 3. Removing
Same tree as exercise 1. For each value: leaf, one child, or two children? Draw the tree after removing it, each time starting from the original tree.

- `10`
- `28`
- `42`: once with the predecessor, once with the successor.

- `10`, leaf: `42(21(, 32(28(26, ), 37)), 64(54(50, 59), 75))`
- `28`, one child (`26` takes its place): `42(21(10, 32(26, 37)), 64(54(50, 59), 75))`
- `42`, two children.
  - Predecessor `37`, a leaf: `37(21(10, 32(28(26, ), )), 64(54(50, 59), 75))`
  - Successor `50`, a leaf: `50(21(10, 32(28(26, ), 37)), 64(54(, 59), 75))`

In [ ]:
check(shape(remove(build(ex1), 10)).equals("42(21(, 32(28(26, ), 37)), 64(54(50, 59), 75))"), "10");
check(shape(remove(build(ex1), 28)).equals("42(21(10, 32(26, 37)), 64(54(50, 59), 75))"), "28");
check(shape(remove(build(ex1), 42)).equals("37(21(10, 32(28(26, ), )), 64(54(50, 59), 75))"), "42, predecessor");

### 4. Iterative contains
Write `contains` without recursion. Does it need a stack, or another structure, to keep track of the branches not explored?

In [ ]:
boolean containsIterative(BTNode<Integer> node, int value) {
    while (node != null) {
        if (value < node.data) node = node.left;
        else if (value > node.data) node = node.right;
        else return true;
    }
    return false;
}

check(containsIterative(root, 58) && !containsIterative(root, 40), "58, 40");
check(!containsIterative(null, 1), "empty tree");

No stack. The search follows one path and never comes back: the branch not taken can never contain the value. Compare `sizeIterative`, which needs every branch.